# Checkpoint 3 — Contest 3 Practice Solutions

Worked answers for Questions 1–8, and a reference program for Question 9.
Each check cell works its answer out again with the tested helper from its unit: a truth table and simplifier from Unit 8, a queue run and a tree build from Unit 9, and the automaton's table and a comparison of two expressions on every short string from Unit 11.
The two program questions are copied into the `assets` folder (`assets/trace5.py` and `assets/trace6.py`) and run.
The Question 9 program is saved as `assets/q9.py`; run it with `python assets/q9.py < input.txt`.
If you missed a question, go back to its unit: Questions 1–2 are Boolean Algebra, 3–4 are Data Structures, 5–6 are What Does This Program Do? – Arrays, and 7–8 are FSAs and Regular Expressions.

## Question 1

Rewrite the middle term with De Morgan, then split one term and combine.

- `~(~A + C)` = `~~A * ~C` = `A * ~C`, so the expression is `A * ~B + A * ~C + ~A * ~B * C`.
- Split `A * ~B` on `C`: `A * ~B` = `A * ~B * C + A * ~B * ~C`.
- `A * ~B * C + ~A * ~B * C` = `(A + ~A) * ~B * C` = `~B * C`.
- `A * ~B * ~C` is already covered by `A * ~C`, so it drops out.
- That leaves `A * ~C + ~B * C`.

Check with the truth table: the expression is true exactly in rows `001, 100, 101, 110`.
`A * ~C` covers `100` and `110`, and `~B * C` covers `001` and `101`.
Both terms are needed (only `A * ~C` covers `110`, and only `~B * C` covers `001`), and neither can lose a letter without covering a false row, so `A * ~C + ~B * C` is the only simplest form.
The question's first term, `A * ~B`, is not needed at all: every row it covers is covered by the other two.

**Answer:** `A * ~C + ~B * C`

In [ ]:
import sys

sys.path.insert(0, "../../units/unit-08-boolean-algebra/assets/verify")
import bool_eval

minimal = bool_eval.minimal_sops("A * ~B + ~(~A + C) + ~A * ~B * C")
assert len(minimal) == 1
assert str(minimal[0]) == "A * ~C + ~B * C"

## Question 2

Use the precedence `~`, then `*`, then `⊕`, then `+`.
So the expression is `(~(A * B) ⊕ C) + (A * ~B * ~C)`.

| A B C | ~(A * B) | ~(A * B) ⊕ C | A * ~B * ~C | result |
|---|---|---|---|---|
| 0 0 0 | 1 | 1 | 0 | 1 |
| 0 0 1 | 1 | 0 | 0 | 0 |
| 0 1 0 | 1 | 1 | 0 | 1 |
| 0 1 1 | 1 | 0 | 0 | 0 |
| 1 0 0 | 1 | 1 | 1 | 1 |
| 1 0 1 | 1 | 0 | 0 | 0 |
| 1 1 0 | 0 | 0 | 0 | 0 |
| 1 1 1 | 0 | 1 | 0 | 1 |

Four rows are true.

**Answer:** `(0,0,0), (0,1,0), (1,0,0), (1,1,1)`

In [ ]:
import sys

sys.path.insert(0, "../../units/unit-08-boolean-algebra/assets/verify")
import bool_eval

assert str(bool_eval.solutions("~(A * B) ⊕ C + A * ~(B + C)")) == "(0,0,0), (0,1,0), (1,0,0), (1,1,1)"

## Question 3

A queue is first in, first out: `POP` takes the item at the front.

| Statement | Queue (front first) | Variable |
|---|---|---|
| `PUSH(6)`, `PUSH(1)`, `PUSH(9)` | 6 1 9 | |
| `X = POP()` | 1 9 | X = 6 |
| `PUSH(4)` | 1 9 4 | |
| `Y = POP()` | 9 4 | Y = 1 |
| `PUSH(X)`, `PUSH(Y)` | 9 4 6 1 | |
| `Z = POP()` | 4 6 1 | Z = 9 |
| `Z = POP()` | 6 1 | Z = 4 |

**Answer:** `4`

In [ ]:
import sys

sys.path.insert(0, "../../units/unit-09-data-structures/assets/verify")
import ds_eval

script = ["PUSH(6)", "PUSH(1)", "PUSH(9)", "X = POP()", "PUSH(4)", "Y = POP()",
          "PUSH(X)", "PUSH(Y)", "Z = POP()", "Z = POP()"]
assert str(ds_eval.run(script, "queue")["vars"]["Z"]) == "4"

## Question 4

Insert `S E P T E M B E R` in order; a letter equal to a node's letter goes left, and the root has depth 0.

| Letter | Path from the root | Depth |
|---|---|---|
| S | root | 0 |
| E | left of S | 1 |
| P | S left, E right | 2 |
| T | right of S | 1 |
| E | S left, E left (equal goes left) | 2 |
| M | S left, E right, P left | 3 |
| B | S left, E left, E left | 3 |
| E | S left, E left, E left, B right | 4 |
| R | S left, E right, P right | 3 |

The internal path length is 0 + 1 + 2 + 1 + 2 + 3 + 3 + 4 + 3 = 19.

**Answer:** `19`

In [ ]:
import sys

sys.path.insert(0, "../../units/unit-09-data-structures/assets/verify")
import ds_eval

assert str(ds_eval.bst(list("SEPTEMBER"))["ipl"]) == "19"

## Question 5

`n` is 7, so the first loop runs for `i` = 0, 1, 2 and swaps each item with its mirror: the list is reversed to `[2, 6, 3, 8, 1, 7, 4]`.

The second loop goes left to right, and whenever an item is smaller than the (already updated) item before it, it adds that item on.

| i | A[i] | A[i - 1] | smaller? | new A[i] |
|---|---|---|---|---|
| 1 | 6 | 2 | no | 6 |
| 2 | 3 | 6 | yes | 9 |
| 3 | 8 | 9 | yes | 17 |
| 4 | 1 | 17 | yes | 18 |
| 5 | 7 | 18 | yes | 25 |
| 6 | 4 | 25 | yes | 29 |

The list ends as `[2, 6, 9, 17, 18, 25, 29]`.
`A[n - 1] - A[0]` is 29 - 2 = 27, and `A[3]` is 17; `print` puts one space between them.

**Answer:** `27 17`

In [ ]:
import subprocess
import sys

result = subprocess.run([sys.executable, "assets/trace5.py"], input='', capture_output=True, text=True)
output = result.stdout.rstrip("\n")
assert str(output) == '27 17'

## Question 6

`A(r, c)` = `(r * c + r) % 5`.
The last loop adds the main-diagonal item `A(k, k)` and takes away the anti-diagonal item `A(k, 5 - k)`.

| k | A(k, k) | A(k, 5 - k) |
|---|---|---|
| 1 | (1 + 1) % 5 = 2 | A(1, 4) = (4 + 1) % 5 = 0 |
| 2 | (4 + 2) % 5 = 1 | A(2, 3) = (6 + 2) % 5 = 3 |
| 3 | (9 + 3) % 5 = 2 | A(3, 2) = (6 + 3) % 5 = 4 |
| 4 | (16 + 4) % 5 = 0 | A(4, 1) = (4 + 4) % 5 = 3 |

`s` = (2 + 1 + 2 + 0) - (0 + 3 + 4 + 3) = 5 - 10 = -5.

**Answer:** `-5`

The check cell runs a Python copy of the program (`assets/trace6.py`): each `FOR ... TO 4` loop becomes `range(1, 5)`, and the array gets an unused row 0 and column 0 so that `A[r][c]` matches `A(r, c)`.

In [ ]:
import subprocess
import sys

result = subprocess.run([sys.executable, "assets/trace6.py"], input='', capture_output=True, text=True)
output = result.stdout.rstrip("\n")
assert str(output) == '-5'

## Question 7

Look at the table: every `a` switches between the pairs `S0`/`S1` and `S2`/`S3`, and every `b` switches between `S0`/`S2` and `S1`/`S3`.
So the automaton only keeps track of whether it has read an odd or even number of `a`s and of `b`s, and `S3` means an odd number of both.

| Option | String | a's | b's | Accepted? |
|---|---|---|---|---|
| A | `ab` | 1 | 1 | yes (S0, S1, S3) |
| B | `aabb` | 2 | 2 | no (ends in S0) |
| C | `ababab` | 3 | 3 | yes |
| D | `bbbab` | 1 | 4 | no (ends in S1) |
| E | `baaab` | 3 | 2 | no (ends in S1) |

**Answer:** `A, C`

In [ ]:
import sys

sys.path.insert(0, "../../units/unit-11-fsas-regular-expressions/assets/verify")
import fsa_eval

table = {
    ("S0", "a"): "S1", ("S0", "b"): "S2",
    ("S1", "a"): "S0", ("S1", "b"): "S3",
    ("S2", "a"): "S3", ("S2", "b"): "S0",
    ("S3", "a"): "S2", ("S3", "b"): "S1",
}
options = {"A": "ab", "B": "aabb", "C": "ababab", "D": "bbbab", "E": "baaab"}
chosen = [label for label in sorted(options) if fsa_eval.run_dfa(table, "S0", {"S3"}, options[label])]
assert str(", ".join(chosen) or "NONE") == "A, C"

## Question 8

`a(ba)*` describes `a`, `aba`, `ababa`, …: strings that start and end with `a` and alternate letters.

- A. `(ab)*a` gives `a`, `aba`, `ababa`, …: the same strings, grouped the other way. Equivalent.
- B. `a(ab)*` gives `aab`, which is not in the set. Not equivalent.
- C. `(abUλ)*a`: repeating "`ab` or nothing" gives the same strings as `(ab)*`, so this is option A again. Equivalent.
- D. `aUa(ba)*ba`: `a(ba)*ba` is `a` followed by one or more `ba`, and adding `a` itself gives zero or more. Equivalent.
- E. `ab*a*` gives `ab`, which is not in the set. Not equivalent.

**Answer:** `A, C, D`

In [ ]:
import sys

sys.path.insert(0, "../../units/unit-11-fsas-regular-expressions/assets/verify")
import fsa_eval

target = "a(ba)*"
options = {"A": "(ab)*a", "B": "a(ab)*", "C": "(abUλ)*a", "D": "aUa(ba)*ba", "E": "ab*a*"}
chosen = [label for label in sorted(options) if fsa_eval.same_language(target, options[label], "ab", 8)]
assert str(", ".join(chosen) or "NONE") == "A, C, D"

## Question 9

**Notice:** use the queue idiom from Unit 9: a list with a `head` index, where "leave the front" is `head = head + 1` and "join the back" is `append`.
Each round moves `K − 1` players from the front to the back, then removes the front player and adds that number to the output.
There is no need to shorten `K` when it is larger than the queue: the moves simply go round the queue more than once, and the list ends with exactly `N × K` entries (250 000 at `500 500`).
The fixtures include one player (`1 1` and `1 500` both give `1`), `K = 1` (the players leave in order, `1 2 3 4 5`), two players with `K = 2` (`2 1`), `K` larger than the queue (`6 7` gives `1 3 6 2 4 5`), `10 2`, the classic `41 3`, and the scale cases `300 17` and `500 500`.

In [ ]:
parts = input().split()
n = int(parts[0])
k = int(parts[1])
queue = []
for player in range(1, n + 1):
    queue.append(player)
head = 0
order = ""
while head < len(queue):
    for step in range(k - 1):
        queue.append(queue[head])
        head = head + 1
    player = queue[head]
    head = head + 1
    if order == "":
        order = str(player)
    else:
        order = order + " " + str(player)
print(order)